In [33]:
import os
import certifi
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
#from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests


In [2]:
%pip install -U tavily-python langchain-community

  Using cached pydantic_settings-2.15.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------- -------------------------- 0.8/2.4 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------  2.4/2.4 MB 6.1 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 5.8 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 7.1 MB/s  0:00:00
Using cached pydantic_settings-2.15.0-py3-none-any.whl (69 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)

  Attempting uninstall: requests

    Found existing installation: requests 2.31.0

    Uninstalling requests-2.31.0:

      Successfully uninstalled requests-2.31.0

   ------------- -------------------------- 2/6 [pydantic-settings]
  Attempting uninstall: langchain-text-splitters
   ------------- ----

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.1.16 requires langchain-community<0.1,>=0.0.32, but you have langchain-community 0.4.2 which is incompatible.
langchain 0.1.16 requires langchain-core<0.2.0,>=0.1.42, but you have langchain-core 1.5.5 which is incompatible.
langchain 0.1.16 requires langchain-text-splitters<0.1,>=0.0.1, but you have langchain-text-splitters 1.1.2 which is incompatible.
langchain 0.1.16 requires langsmith<0.2.0,>=0.1.17, but you have langsmith 0.11.0 which is incompatible.


In [4]:
%pip install -U langchain langchain-google-genai langchain-community tavily-python

  Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.4.2-py3-none-any.whl.metadata (3.6 kB)
  Using cached ormsgpack-1.12.2-cp311-cp311-win_amd64.whl.metadata (3.3 kB)
  Using cached websockets-15.0.1-cp311-cp311-win_amd64.whl.metadata (7.0 kB)
Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl (56 kB)
Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl (41 kB)
Using cached langgraph_sdk-0.4.2-py3-none-any.whl (160 kB)
Using cached websockets-15.0.1-cp311-cp311-win_amd64.whl (176 kB)
Using cached ormsgpack-1.12.2-cp311-cp311-win_amd64.whl (117 kB)

  Attempting uninstall: websockets

    Found existing installation: websockets 16.1.1

    Uninstalling websockets-16.1.1:

      Successfully uninstalled websockets-16.1.1

   ---------------------------------------- 0/7 [websockets]
   ---------------------------------------- 0/7 [websockets]
   ------

  You can safely remove it manually.


In [6]:
import langchain
from langchain.agents import create_agent

In [34]:
#=====================
# Load environment variables
#=====================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_API_KEY = os.getenv("Tavily_API_KEY")
WEATHERSTACK_API_KEY=os.getenv("WEATHERSTACK_API_KEY")


In [8]:
search_tool=TavilySearchResults(max_results=4)


C:\Users\veerh\AppData\Local\Temp\ipykernel_13404\1539868581.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool=TavilySearchResults(max_results=4)


In [ ]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [ ]:
result=search_tool.invoke("Find the capital of India"
        "and then find its current weather."")
result

[{'title': 'Prime Minister of India - Wikipedia',
  'url': 'https://en.wikipedia.org/wiki/Prime_Minister_of_India',
  'content': 'The longest-serving prime minister was the first prime minister, Jawaharlal Nehru, whose tenure lasted 16 years and 286 days. His premiership was followed by Lal Bahadur Shastri\'s short tenure and Indira Gandhi\'s 11- and 4-year-long tenures, with both politicians belonging to the Indian National Congress. After Indira Gandhi\'s assassination, her son Rajiv Gandhi took charge until 1989, when a decade with five unstable governments began. This was followed by the full terms of P. V. Narasimha Rao, Atal Bihari Vajpayee, Manmohan Singh, and Narendra Modi, who is the current prime minister of India, serving since 26 May 2014. He is the first non-Congress leader to become PM after consecutive general elections and secure a third successive term (2014, 2019, 2024). The first prime minister to do [...] | _Bhārata Gaṇarājya kē Pradhānamantrī_ (ISO) |\n| Image 5 Lo

In [19]:
#================
#LLM
#================
llm=ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0,
    api_key=GEMINI_API_KEY
)

In [20]:
import os
from dotenv import load_dotenv

load_dotenv()

print("API key loaded:", bool(os.getenv("GEMINI_API_KEY")))

API key loaded: True


In [22]:
response = llm.invoke("WHO is current CM Of UP?")
print(response.content)

c:\Users\veerh\anaconda3\envs\langagent\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The current Chief Minister of Uttar Pradesh is **Yogi Adityanath**. He has been in office since March 19, 2017.', 'extras': {'signature': 'Es4ECssEARFNMg+gU/NJmrmur0VFsSHeh2q+6YxY/gzyzGRrCRsGMORSs1ZGvEzjtzoqn/cYQJ0qqyVLrW+XUP8bJMeWDTyeS9H/5kI5dfW6AR89dE9tZ3LHd7QhTSRJJZ9QsppfcFrNzgbkg2c4HRPCHLoIi580KVA726issbr1khEG35J3KWLvHlQiOO3Y4cULFZlpfpXFX2r16/ZRl8RAwkRlbK9tLAce+oL5lRFBhGwPODvnr5LPadcrA2HutFRbiTuvHWhDWT34tdA7MpsF7AKd7CKcgxb3WfBYqIicTFYYg65I2TVeUmEBWohVchQLocv6qpZGjZIotJaSe8BCz28PXOKsb6zFJQ9QatMSI9Ob5mHvTQlcQWMmsxEBt5+GkkBKSFGUXBPvofx8lUDl3ZksBw1wNcRN4wPkUQ2esTtf5kTJFW22jIVqYKsjBJyix9QCH9icxbkOjboFcbzWwjGRC8aIxt69mCa91Xz4N/dTvw/BB0zExYR8vCva+Ar9KFW9XOGq2y58mlh8QY0qsJRdbRPQMdWPVlvJGrU7EPpik3b29g5EwN5k89Rbuos2K7V14r5EzSWkH7d9CkjM7/v/7GPGc35G6xt1UJRCGraZ2znvihJi3bhC3MLInKDi4lfLxDVfWZZW6u3K/lSrY84eH4Br/DNu+810uJZP+Us0W3GMeomPBtZJfwPpk8/GJ1OPuHP2FkU4yjN7RnSMElXcUDD2do0V59b5PU9+xo5AJLIw55Oddc7798Ksg9IyFrr09979WJ44ZEmNkyw='}}]


In [23]:
#=======
#prompt
#=======

prompt=hub.pull("hwchase17/react")
prompt


c:\Users\veerh\anaconda3\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
c:\Users\veerh\anaconda3\envs\langagent\Lib\site-packages\langchain\hub.py:87: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.


PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [ ]:
#=======
#tools
#=======

tools=[search_tool,get_weather_data]

In [29]:
#=========
#create agent
#=========

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Use the search tool when you need current information."
)

In [ ]:
##############
# run
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Find the capital of India and then find its current weather ."
        }
    ]
})

print(response["messages"][-1].content)

c:\Users\veerh\anaconda3\envs\langagent\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\veerh\anaconda3\envs\langagent\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\veerh\anaconda3\envs\langagent\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\veerh\anaconda3\envs\langagent\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperatu

[{'type': 'text', 'text': '### Capital of India\nThe capital of India is **New Delhi**.\n\n---\n\n### Current Weather in Delhi (New Delhi)\n* **Temperature:** ~33°C to 35°C (91°F to 95°F)\n* **Conditions:** Partly cloudy / clear with occasional patchy rain showers (monsoon season)\n* **Humidity:** ~55% – 65%\n* **Wind:** ~10–20 km/h (W/WSW)', 'extras': {'signature': 'Er4ECrsEARFNMg/7SKH+eZoLgxQ5LgW4lXW0FciWFdcLvVSoKnsPR5F8Fa+pHzCo/5oRc54L6TEvC9RRdG6drjPVqY99+fl32ZkxMrDFZ26Gp01XUzDom/b5H2aNTXkST7Q1woAeRAt6BwsmpaYtd5qrIXW9b38R4y7ZJtWdjGzwu8dZ/6/2ZK6ZlH5pwcvyIPAUYxtOJsywemdow4FG7Y3eoZMFKXMzedQR9XbYC0HQAsXz+iukkiY5S2ku0fnio/MGZPePO06rWDZVHG2h5nVlQX9E+5BjHjLy+vCgvA8XitGFT1RlAUxkL10K6izSGP4kpYhkYdpyND8bxcrdIpAAAW3gVu0DEOcohtFQlrfth9uxiGIKQeCofEUqpHb8b5zS183S7Ts/e60HvNBq6waNwB+TKEg6yFgom4aNwUEKExBDQESFmDD9glcvRg8JDgwE6P/EJvE2kVRPZCPsGFb+ELZQQZw/PPpPURYF3iXEUa+hWViVMhfxRkdS117H4Wr91W8XApp+x4qroWB8UpBOv6f7Jb3UPl2cHoBIamjwqktAHGkFrpEidqK5NAaw2z63kKCbLvXNf+n2zHRb5m6NVCqqRgPsiU5zh13bCi331j04g4hc33